# 02 - Preprocessing & Feature Engineering
## Đề tài 11: Phân tích đánh giá khách sạn & chủ đề dịch vụ

**Mục tiêu:**
- Tiền xử lý văn bản (làm sạch, stopwords, stemming)
- Trích xuất đặc trưng TF-IDF
- Thêm đặc trưng thống kê

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import load_data
from src.data.cleaner import DataCleaner
from src.features.builder import FeatureBuilder

[WARNING] HDBSCAN not available. Install with: pip install hdbscan


## 1. Load Data

In [2]:
# Load raw data
df = load_data(n_rows=2000, config_path='../configs/params.yaml')
print(f"Raw data shape: {df.shape}")
df.head()

[INFO] File not found at data\raw\hotel_reviews.csv
[INFO] Generating sample hotel reviews data...
[INFO] Generating 2000 sample hotel reviews...
[INFO] Generated 2000 sample reviews
[INFO] Rating distribution:
rating
1    147
2    208
3    579
4    477
5    589
Name: count, dtype: int64
Raw data shape: (2000, 6)


,review_text,rating,sentiment,reviewer_name,date,hotel_name
0,The room was nice but the noise from the stree...,4,mixed,Jennifer L.,2020-07-09,Royal Gardens Hotel
1,"Overall, perfect getaway! the hotel exceeded ...",4,positive,Laura B.,2022-06-07,City Center Suites
2,The room was clean but small. The location was...,3,neutral,Kevin J.,2023-08-15,Ocean Breeze Resort
3,"The room was spotless, the bed was comfortable...",5,positive,James W.,2020-02-12,Luxury Stay Hotel
4,"Considering everything, the room was clean bu...",3,neutral,Kevin J.,2020-07-08,Mountain View Inn


## 2. Text Preprocessing

In [3]:
# Initialize cleaner
cleaner = DataCleaner()

# Clean data
df_cleaned, stats = cleaner.clean(df, text_column='review_text', rating_column='rating')

print(f"\nCleaned data shape: {df_cleaned.shape}")
print(f"\nPreprocessing stats: {stats}")

DATA CLEANING PROCESS

[Step 1/5] Basic data cleaning...
  - Original rows: 2000

[Step 2/5] Handling missing values...

[Step 3/5] Text preprocessing...
[INFO] Cleaning text...

[Step 4/5] Validating ratings...

[Step 5/5] Adding derived features...

CLEANING SUMMARY
Original rows: 2000
Final rows: 2000
Rows removed: 0 (0.0%)
Columns: ['review_text', 'rating', 'sentiment', 'reviewer_name', 'date', 'hotel_name', 'original_text', 'original_length', 'original_word_count', 'cleaned_text', 'cleaned_length', 'cleaned_word_count', 'length_reduction', 'word_reduction', 'review_length', 'word_count', 'avg_word_length', 'sentence_count', 'exclamation_count', 'question_count', 'uppercase_count', 'uppercase_ratio']

Cleaned data shape: (2000, 22)

Preprocessing stats: {'original_rows': 2000, 'final_rows': 2000, 'rows_removed': 0, 'removal_rate': '0.0%', 'text_cleaning_stats': {'original_rows': 2000, 'cleaned_rows': 2000, 'removed_duplicates': 0, 'removed_missing': np.int64(0), 'avg_text_length_be

## 3. Before/After Comparison

In [4]:
# Compare before and after
comparison = cleaner.get_before_after_comparison(df, df_cleaned)
print("Before/After Comparison:")
for k, v in comparison.items():
    print(f"  {k}: {v}")

Before/After Comparison:
  n_rows: {'before': 2000, 'after': 2000, 'difference': 0}
  avg_review_length: {'before': 78.299, 'after': 78.299, 'reduction': 0.0}
  avg_word_count: {'before': 12.2995, 'after': 12.2995}
  rating_distribution: {'before': {5: 589, 3: 579, 4: 477, 2: 208, 1: 147}, 'after': {5: 589, 3: 579, 4: 477, 2: 208, 1: 147}}


## 4. Feature Engineering

In [5]:
# Initialize feature builder
feature_builder = FeatureBuilder()

# Build TF-IDF features
cleaned_texts = df_cleaned['cleaned_text'].astype(str).tolist()
tfidf_matrix = feature_builder.build_tfidf_features(cleaned_texts, fit=True)

# Build statistical features
stat_features = feature_builder.build_statistical_features(df_cleaned, text_column='review_text')

# Build aspect features
aspect_features, aspect_counts = feature_builder.build_aspect_features(df_cleaned, text_column='review_text')

print(f"\nTF-IDF shape: {tfidf_matrix.shape}")
print(f"Statistical features shape: {stat_features.shape}")
print(f"Aspect features shape: {aspect_features.shape}")
print(f"\nAspect counts: {aspect_counts}")

[INFO] TF-IDF fitted with 5000 features, shape: (2000, 621)

TF-IDF shape: (2000, 621)
Statistical features shape: (2000, 20)
Aspect features shape: (2000, 28)

Aspect counts: {'room': np.int64(919), 'service': np.int64(845), 'location': np.int64(502), 'food': np.int64(200), 'price': np.int64(329), 'amenities': np.int64(367), 'cleanliness': np.int64(526), 'noise': np.int64(88)}


## 5. Save Processed Data

In [6]:
# Save cleaned data
df_cleaned.to_csv('../data/processed/cleaned_data.csv', index=False)
print("Saved: ../data/processed/cleaned_data.csv")

# Save features
np.save('../data/processed/tfidf_features.npy', tfidf_matrix.toarray())
np.save('../data/processed/stat_features.npy', stat_features)
np.save('../data/processed/aspect_features.npy', aspect_features)
print("Saved features to ../data/processed/")

Saved: ../data/processed/cleaned_data.csv
Saved features to ../data/processed/
